In [1]:
# ---------------------------------------------------------------------
# CELL 1: SETUP
# Master input inventory + readiness notebook
# scripts/master_inputs/master.ipynb
# ---------------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
import json
import warnings

import numpy as np
import pandas as pd
import rasterio

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------------------------------------------------------------
# ROOT PATHS
# ---------------------------------------------------------------------
PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RASTERS_DIR = PROJECT_ROOT / "rasters"

# Historical / reference / corrected inputs
ERA5_DIR = DATA_DIR / "simulated" / "era5_land"

CMIP6_CORRECTED_DIR = RASTERS_DIR / "corrected" / "cmip6"
PROXY_CORRECTED_DIR = RASTERS_DIR / "corrected" / "proxies"

# Manifest outputs
ARTIFACTS_DIR = DATA_DIR / "artifacts"
MANIFESTS_DIR = ARTIFACTS_DIR / "manifests"
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID = f"master_inputs_{RUN_TS}"

SCENARIOS = ["ssp245", "ssp370", "ssp585"]

HIST_START = "198109"
HIST_END = "202602"

# TWS observed/proxy historical ends earlier than NDVI
TWS_HIST_END = "202409"

FUTURE_START = "202601"
FUTURE_END = "205012"

# ---------------------------------------------------------------------
# VARIABLES
# ---------------------------------------------------------------------
BASE_VARIABLES = {
    "t2m": ERA5_DIR / "t2m",
    "d2m": ERA5_DIR / "d2m",
    "sm": ERA5_DIR / "sm",
    "pet": ERA5_DIR / "pet",
}

PROXY_HISTORICAL = {
    "ndvi": PROXY_CORRECTED_DIR / "ndvi" / "historical",
    "tws": PROXY_CORRECTED_DIR / "tws" / "historical",
}

FUTURE_CMIP6 = {
    "tas": "tas",
    "d2m": "d2m",
    "sm": "sm",
    "pet": "pet",
}

FUTURE_PROXIES = {
    "ndvi": "ndvi",
    "tws": "tws",
}

MASTER_MANIFEST_PATH = MANIFESTS_DIR / f"{RUN_ID}_master_manifest.json"
READINESS_MANIFEST_PATH = MANIFESTS_DIR / f"{RUN_ID}_readiness_manifest.json"

# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------
def extract_yyyymm(path: Path):
    stem = path.stem
    for token in stem.split("_"):
        if token.isdigit() and len(token) == 6:
            return token
    return None


def index_by_yyyymm(folder: Path, pattern: str = "*.tif"):
    out = {}
    if not folder.exists():
        return out
    for fp in sorted(folder.glob(pattern)):
        ym = extract_yyyymm(fp)
        if ym is not None:
            out[ym] = fp
    return out


def month_range(start: str, end: str):
    months = []
    y = int(start[:4])
    m = int(start[4:])

    while True:
        ym = f"{y:04d}{m:02d}"
        months.append(ym)

        if ym == end:
            break

        m += 1
        if m > 12:
            m = 1
            y += 1

    return months


def save_json(obj: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


print("=" * 60)
print("MASTER INPUTS SETUP")
print("=" * 60)
print(f"PROJECT ROOT           : {PROJECT_ROOT}")
print(f"RUN ID                 : {RUN_ID}")
print(f"HIST PERIOD            : {HIST_START} -> {HIST_END}")
print(f"TWS HIST END           : {TWS_HIST_END}")
print(f"FUTURE PERIOD          : {FUTURE_START} -> {FUTURE_END}")
print(f"MASTER MANIFEST PATH   : {MASTER_MANIFEST_PATH}")
print(f"READINESS MANIFEST PATH: {READINESS_MANIFEST_PATH}")

MASTER INPUTS SETUP
PROJECT ROOT           : C:\Projects\Infer RozviDrought
RUN ID                 : master_inputs_20260323T084703Z
HIST PERIOD            : 198109 -> 202602
TWS HIST END           : 202409
FUTURE PERIOD          : 202601 -> 205012
MASTER MANIFEST PATH   : C:\Projects\Infer RozviDrought\data\artifacts\manifests\master_inputs_20260323T084703Z_master_manifest.json
READINESS MANIFEST PATH: C:\Projects\Infer RozviDrought\data\artifacts\manifests\master_inputs_20260323T084703Z_readiness_manifest.json


In [2]:
# ---------------------------------------------------------------------
# CELL 2: INVENTORY ALL INPUTS
# Build a full inventory of historical and future inputs.
# ---------------------------------------------------------------------

print("=" * 60)
print("MASTER INPUT INVENTORY")
print("=" * 60)

master_inventory = {
    "run_id": RUN_ID,
    "historical": {},
    "future": {},
}

# ------------------------------------------------------------
# Historical base variables
# ------------------------------------------------------------
print("\nHISTORICAL BASE VARIABLES")
print("-" * 60)

for var, folder in BASE_VARIABLES.items():
    idx = index_by_yyyymm(folder)

    info = {
        "path": str(folder),
        "months_found": len(idx),
        "first_month": min(idx.keys()) if idx else None,
        "last_month": max(idx.keys()) if idx else None,
    }

    master_inventory["historical"][var] = info
    print({var: info})

# ------------------------------------------------------------
# Historical proxy variables
# ------------------------------------------------------------
print("\nHISTORICAL PROXY VARIABLES")
print("-" * 60)

for var, folder in PROXY_HISTORICAL.items():
    idx = index_by_yyyymm(folder)

    info = {
        "path": str(folder),
        "months_found": len(idx),
        "first_month": min(idx.keys()) if idx else None,
        "last_month": max(idx.keys()) if idx else None,
    }

    master_inventory["historical"][var] = info
    print({var: info})

# ------------------------------------------------------------
# Future CMIP6 corrected variables by scenario
# ------------------------------------------------------------
print("\nFUTURE CMIP6 VARIABLES")
print("-" * 60)

for scenario in SCENARIOS:
    master_inventory["future"][scenario] = {}

    for var, subfolder in FUTURE_CMIP6.items():
        folder = CMIP6_CORRECTED_DIR / scenario / subfolder
        idx = index_by_yyyymm(folder)

        info = {
            "path": str(folder),
            "months_found": len(idx),
            "first_month": min(idx.keys()) if idx else None,
            "last_month": max(idx.keys()) if idx else None,
        }

        master_inventory["future"][scenario][var] = info
        print({"scenario": scenario, "variable": var, **info})

# ------------------------------------------------------------
# Future proxy variables by scenario
# ------------------------------------------------------------
print("\nFUTURE PROXY VARIABLES")
print("-" * 60)

for scenario in SCENARIOS:
    for var, subfolder in FUTURE_PROXIES.items():
        folder = PROXY_CORRECTED_DIR / subfolder / scenario
        idx = index_by_yyyymm(folder)

        info = {
            "path": str(folder),
            "months_found": len(idx),
            "first_month": min(idx.keys()) if idx else None,
            "last_month": max(idx.keys()) if idx else None,
        }

        master_inventory["future"][scenario][var] = info
        print({"scenario": scenario, "variable": var, **info})

save_json(master_inventory, MASTER_MANIFEST_PATH)

print("\n" + "=" * 60)
print("MASTER INVENTORY SAVED")
print("=" * 60)
print(f"Path : {MASTER_MANIFEST_PATH}")

MASTER INPUT INVENTORY

HISTORICAL BASE VARIABLES
------------------------------------------------------------
{'t2m': {'path': 'C:\\Projects\\Infer RozviDrought\\data\\simulated\\era5_land\\t2m', 'months_found': 523, 'first_month': '198109', 'last_month': '202603'}}
{'d2m': {'path': 'C:\\Projects\\Infer RozviDrought\\data\\simulated\\era5_land\\d2m', 'months_found': 523, 'first_month': '198109', 'last_month': '202603'}}
{'sm': {'path': 'C:\\Projects\\Infer RozviDrought\\data\\simulated\\era5_land\\sm', 'months_found': 555, 'first_month': '198001', 'last_month': '202603'}}
{'pet': {'path': 'C:\\Projects\\Infer RozviDrought\\data\\simulated\\era5_land\\pet', 'months_found': 523, 'first_month': '198109', 'last_month': '202603'}}

HISTORICAL PROXY VARIABLES
------------------------------------------------------------
{'ndvi': {'path': 'C:\\Projects\\Infer RozviDrought\\rasters\\corrected\\proxies\\ndvi\\historical', 'months_found': 534, 'first_month': '198109', 'last_month': '202602'}}
{'

In [3]:
# ---------------------------------------------------------------------
# CELL 3: GLOBAL INFERENCE READINESS CHECK
# Robust missing-month detection (.tif + .nc)
# ---------------------------------------------------------------------

import xarray as xr

print("=" * 60)
print("GLOBAL INFERENCE READINESS CHECK")
print("=" * 60)

readiness = {
    "run_id": RUN_ID,
    "historical": {},
    "future": {},
    "status": "unknown",
}

# ------------------------------------------------------------
# HELPER: read months from NetCDF files
# First try filename, then fall back to internal time coord
# ------------------------------------------------------------
def index_nc_time_months(folder: Path):

    months_found = set()

    if not folder.exists():
        return months_found

    for fp in sorted(folder.glob("*.nc")):

        # Prefer filename-based month extraction
        ym = extract_yyyymm(fp)
        if ym is not None:
            months_found.add(ym)
            continue

        # Fallback: inspect time coordinate
        try:
            ds = xr.open_dataset(fp)

            if "time" in ds.coords:
                times = pd.to_datetime(ds["time"].values)

                for t in times:
                    months_found.add(
                        f"{t.year}{t.month:02d}"
                    )

            ds.close()

        except Exception:
            continue

    return months_found


# ------------------------------------------------------------
# HELPER: combine tif + nc months
# ------------------------------------------------------------
def combined_month_index(folder: Path):

    tif_months = set(
        index_by_yyyymm(folder).keys()
    )

    nc_months = index_nc_time_months(folder)

    combined = tif_months | nc_months

    return tif_months, nc_months, combined


# ------------------------------------------------------------
# EXPECTED MONTH RANGES
# ------------------------------------------------------------
hist_expected = set(
    month_range(HIST_START, HIST_END)
)

tws_expected = set(
    month_range(HIST_START, TWS_HIST_END)
)

future_expected = set(
    month_range(FUTURE_START, FUTURE_END)
)

# ------------------------------------------------------------
# HISTORICAL READINESS
# ------------------------------------------------------------
print("\nHISTORICAL READINESS")
print("-" * 60)

for var, folder in BASE_VARIABLES.items():

    tif_found, nc_found, found = combined_month_index(folder)

    missing = sorted(
        hist_expected - found
    )

    info = {
        "tif_months_found": len(tif_found),
        "nc_months_found": len(nc_found),
        "combined_months_found": len(found),
        "missing_months": missing,
        "missing_count": len(missing),
    }

    readiness["historical"][var] = info

    print({
        "variable": var,
        "tif": len(tif_found),
        "nc": len(nc_found),
        "missing": len(missing),
    })


# ------------------------------------------------------------
# HISTORICAL PROXIES
# ------------------------------------------------------------
for var, folder in PROXY_HISTORICAL.items():

    tif_found, nc_found, found = combined_month_index(folder)

    expected = (
        tws_expected
        if var == "tws"
        else hist_expected
    )

    missing = sorted(
        expected - found
    )

    info = {
        "tif_months_found": len(tif_found),
        "nc_months_found": len(nc_found),
        "combined_months_found": len(found),
        "missing_months": missing,
        "missing_count": len(missing),
    }

    readiness["historical"][var] = info

    print({
        "variable": var,
        "tif": len(tif_found),
        "nc": len(nc_found),
        "missing": len(missing),
    })


# ------------------------------------------------------------
# FUTURE READINESS
# ------------------------------------------------------------
print("\nFUTURE READINESS")
print("-" * 60)

future_ok = True

for scenario in SCENARIOS:

    readiness["future"][scenario] = {}

    for var in ["tas", "d2m", "sm", "pet"]:

        folder = (
            CMIP6_CORRECTED_DIR
            / scenario
            / var
        )

        tif_found, nc_found, found = combined_month_index(folder)

        missing = sorted(
            future_expected - found
        )

        if missing:
            future_ok = False

        info = {
            "tif_months_found": len(tif_found),
            "nc_months_found": len(nc_found),
            "combined_months_found": len(found),
            "missing_months": missing,
            "missing_count": len(missing),
        }

        readiness["future"][scenario][var] = info

        print({
            "scenario": scenario,
            "variable": var,
            "tif": len(tif_found),
            "nc": len(nc_found),
            "missing": len(missing),
        })

    for var in ["ndvi", "tws"]:

        folder = (
            PROXY_CORRECTED_DIR
            / var
            / scenario
        )

        tif_found, nc_found, found = combined_month_index(folder)

        missing = sorted(
            future_expected - found
        )

        if missing:
            future_ok = False

        info = {
            "tif_months_found": len(tif_found),
            "nc_months_found": len(nc_found),
            "combined_months_found": len(found),
            "missing_months": missing,
            "missing_count": len(missing),
        }

        readiness["future"][scenario][var] = info

        print({
            "scenario": scenario,
            "variable": var,
            "tif": len(tif_found),
            "nc": len(nc_found),
            "missing": len(missing),
        })


# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------
all_hist_ok = all(
    v["missing_count"] == 0
    for v in readiness["historical"].values()
)

readiness["status"] = (
    "READY"
    if (all_hist_ok and future_ok)
    else "NOT_READY"
)

save_json(
    readiness,
    READINESS_MANIFEST_PATH
)

print("\n" + "=" * 60)
print("READINESS STATUS")
print("=" * 60)
print(readiness["status"])
print("Manifest saved :", READINESS_MANIFEST_PATH)

GLOBAL INFERENCE READINESS CHECK

HISTORICAL READINESS
------------------------------------------------------------
{'variable': 't2m', 'tif': 523, 'nc': 32, 'missing': 0}
{'variable': 'd2m', 'tif': 523, 'nc': 32, 'missing': 0}
{'variable': 'sm', 'tif': 555, 'nc': 0, 'missing': 0}
{'variable': 'pet', 'tif': 523, 'nc': 32, 'missing': 0}
{'variable': 'ndvi', 'tif': 534, 'nc': 0, 'missing': 0}
{'variable': 'tws', 'tif': 517, 'nc': 0, 'missing': 0}

FUTURE READINESS
------------------------------------------------------------
{'scenario': 'ssp245', 'variable': 'tas', 'tif': 300, 'nc': 0, 'missing': 0}
{'scenario': 'ssp245', 'variable': 'd2m', 'tif': 300, 'nc': 0, 'missing': 0}
{'scenario': 'ssp245', 'variable': 'sm', 'tif': 300, 'nc': 0, 'missing': 0}
{'scenario': 'ssp245', 'variable': 'pet', 'tif': 300, 'nc': 0, 'missing': 0}
{'scenario': 'ssp245', 'variable': 'ndvi', 'tif': 300, 'nc': 0, 'missing': 0}
{'scenario': 'ssp245', 'variable': 'tws', 'tif': 300, 'nc': 0, 'missing': 0}
{'scenario

In [17]:
# ---------------------------------------------------------------------
# CELL 4: SHOW THE ACTUAL MISSING MONTHS
# ---------------------------------------------------------------------

print("=" * 60)
print("ACTUAL MISSING MONTHS")
print("=" * 60)

for var in ["t2m", "d2m", "pet"]:
    print(f"\nHISTORICAL {var.upper()}")
    print(readiness["historical"][var]["missing_months"])

for scenario in SCENARIOS:
    print(f"\nFUTURE NDVI {scenario}")
    print(readiness["future"][scenario]["ndvi"]["missing_months"])

ACTUAL MISSING MONTHS

HISTORICAL T2M
[]

HISTORICAL D2M
[]

HISTORICAL PET
[]

FUTURE NDVI ssp245
[]

FUTURE NDVI ssp370
[]

FUTURE NDVI ssp585
[]
